# AutoPose — GigaPose-style 6-DoF Object Pose Estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunachaleshwaran/autopose/blob/main/notebooks/autopose_colab.ipynb)

## Pipeline Overview

| Stage | What | How |
|---|---|---|
| **1 — Template retrieval** | Predict azimuth + elevation (2 DoF) | Nearest-neighbour search in Fae (DINOv2) feature space over 162 icosphere templates |
| **2 — In-plane refinement** | Predict in-plane rotation + translation + scale (4 DoF) | Fist (ResNet18 + MLP heads) regresses (cos α, sin α, tx, ty, s) |

Azimuth & elevation are **retrieved, not regressed** — the best-matching template's pose from `templates/poses.json` is the answer.

---

**Before running:** upload your local `templates/` folder (162 PNGs + `poses.json`) to Google Drive at `MyDrive/autopose/templates/`.

## 0 — Setup

In [ ]:
# Clone repo (skip if already present)
import os

REPO_DIR = "/content/autopose"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/arunachaleshwaran/autopose.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

In [ ]:
# System dependencies (pyrender needs osmesa for headless rendering)
import os
import subprocess

subprocess.run(["apt-get", "install", "-qq", "libosmesa6"], check=True)
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
print("System deps installed.")

In [ ]:
# Install pip packages from environment.yml (pip: section)
import subprocess
import yaml

with open("environment.yml") as f:
    env = yaml.safe_load(f)

pip_pkgs = next(
    d["pip"] for d in env["dependencies"] if isinstance(d, dict) and "pip" in d
)

print(f"Installing {len(pip_pkgs)} packages from environment.yml pip section...")
subprocess.check_call(["pip", "install", "-q"] + pip_pkgs)
print("Done.")

In [ ]:
# Install conda-listed packages not already in Colab base
# (torch/numpy/scipy/pandas/matplotlib are pre-installed on Colab)
!pip install -q \
    opencv-python-headless \
    trimesh \
    networkx \
    shapely \
    imageio \
    scikit-image \
    tqdm \
    ipywidgets
print("Conda-equivalent packages installed.")

In [ ]:
# Mount Google Drive and symlink templates/
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/autopose"
TEMPLATES_SRC = f"{DRIVE_ROOT}/templates"
TEMPLATES_DST = "/content/autopose/templates"

assert os.path.isdir(TEMPLATES_SRC), (
    f"templates/ not found at {TEMPLATES_SRC}. "
    "Upload your templates/ folder to Google Drive at MyDrive/autopose/templates/"
)

# Symlink so all downstream code uses templates/ as a normal relative path
if not os.path.islink(TEMPLATES_DST):
    os.symlink(TEMPLATES_SRC, TEMPLATES_DST)

import json
poses = json.loads(open("templates/poses.json").read())
print(f"templates/ linked. Found {len(poses)} pose entries.")

In [ ]:
# Verify GPU and key imports
import torch
import timm
import lightning
import transformers
import kornia
import roma
import trimesh
import pyrender

print(f"torch      {torch.__version__}")
print(f"lightning  {lightning.__version__}")
print(f"timm       {timm.__version__}")
print(f"transformers {transformers.__version__}")
print()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — go to Runtime > Change runtime type > GPU")

---

## 1 — Load Template Data

`templates/poses.json` stores 162 entries, one per icosphere vertex. Each entry has:
- `azimuth_deg` / `elevation_deg` — the viewpoint angle
- `R_world_to_cam_3x3` — rotation matrix (world → camera)
- `t_cam_to_world_3` — camera position in world space
- `file` — corresponding PNG (`tmpl_XXXX.png`)

Azimuth + elevation together define the **out-of-plane rotation** (2 of 6 DoF).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

poses = json.loads(open("templates/poses.json").read())
df = pd.DataFrame(poses)
print(f"Loaded {len(df)} templates")
display(df[["frame", "file", "azimuth_deg", "elevation_deg"]].head())

def load_template_rgb(path):
    """Load RGBA template PNG and composite onto white background."""
    img = Image.open(path).convert("RGBA")
    bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
    bg.paste(img, mask=img.split()[3])
    return bg.convert("RGB")

templates_rgb = [load_template_rgb(f"templates/{row['file']}") for _, row in df.iterrows()]
print(f"Loaded {len(templates_rgb)} template images  size={templates_rgb[0].size}")

# Viewpoint sphere
fig = plt.figure(figsize=(10, 4))
ax3d = fig.add_subplot(121, projection="3d")
xyz = np.array(df["viewpoint_xyz"].tolist())
ax3d.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], s=10, c=df["elevation_deg"], cmap="coolwarm")
ax3d.set_title("162 icosphere viewpoints")

# Az/El scatter
ax2d = fig.add_subplot(122)
sc = ax2d.scatter(df["azimuth_deg"], df["elevation_deg"], s=15, c=df["elevation_deg"], cmap="coolwarm")
ax2d.set_xlabel("Azimuth (°)"); ax2d.set_ylabel("Elevation (°)")
ax2d.set_title("Azimuth vs Elevation"); plt.colorbar(sc, ax=ax2d, label="elevation °")
plt.tight_layout(); plt.show()

# Grid of 12 sample templates
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
indices = np.linspace(0, 161, 12, dtype=int)
for ax, i in zip(axes.flat, indices):
    ax.imshow(templates_rgb[i])
    ax.set_title(f"az={df.iloc[i]['azimuth_deg']:.0f}°\nel={df.iloc[i]['elevation_deg']:.0f}°", fontsize=7)
    ax.axis("off")
plt.suptitle("Sample templates"); plt.tight_layout(); plt.show()

---

## 2 — Template Feature Bank (Fae)

We use a **pretrained DINOv2 ViT-L/14** (no fine-tuning) as Fae. The paper fine-tunes it with InfoNCE loss, but pretrained DINOv2 still retrieves meaningful viewpoints and is immediately runnable.

- Input: 224×224 RGB image
- Patch size: 14×14 px → **16×16 = 256 patches** per image
- Output per patch: **1024-dim** feature vector
- Total template bank shape: **(162, 256, 1024)**

All 162 template features are precomputed once and saved to `templates/template_features.pt`.

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# DINOv2 ViT-L/14 — 1024-dim patch tokens, matches paper's 16×16×1024 output
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14")
dino = dino.eval().to(DEVICE)
print("DINOv2 ViT-L/14 loaded")

# ImageNet normalisation (same stats DINOv2 was pretrained with)
preprocess = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Batch forward pass over all 162 templates
BATCH = 16
all_feats = []

with torch.no_grad():
    for i in range(0, len(templates_rgb), BATCH):
        batch = torch.stack([preprocess(img) for img in templates_rgb[i:i+BATCH]]).to(DEVICE)
        # x_norm_patchtokens: (B, num_patches, dim) = (B, 256, 1024)
        out = dino.forward_features(batch)["x_norm_patchtokens"]
        all_feats.append(out.cpu())
        print(f"  batch {i//BATCH + 1}/{-(-len(templates_rgb)//BATCH)}  shape={tuple(out.shape)}", end="\r")

template_feats = torch.cat(all_feats, dim=0)          # (162, 256, 1024)
template_feats = F.normalize(template_feats, dim=-1)   # L2-normalise for cosine sim

torch.save(template_feats, "templates/template_features.pt")
print(f"\nSaved templates/template_features.pt  shape={tuple(template_feats.shape)}")

---

## 3 — Azimuth & Elevation Prediction (Stage 1)

**How it works:**
1. Upload a real photo of your 3DBenchy and set a manual crop box
2. The crop is encoded by DINOv2 → `(256, 1024)` patch features
3. Each query patch finds its nearest match across all `162 × 256` template patches (cosine similarity)
4. Templates are scored by the mean similarity of patches assigned to them (threshold 0.5)
5. The top-K templates' `azimuth_deg` / `elevation_deg` from `poses.json` **are** the prediction

In [ ]:
### 3a — Upload pre-cropped photo

from google.colab import files
import io

uploaded = files.upload()
fname = next(iter(uploaded))
query_img = Image.open(io.BytesIO(uploaded[fname])).convert("RGB")
print(f"Uploaded: {fname}  original size: {query_img.size[0]}×{query_img.size[1]}px")

def resize_pad(img, size=224, fill=(255, 255, 255)):
    """Resize longest side to `size`, pad short side to square with `fill`."""
    img = img.copy()
    img.thumbnail((size, size), Image.BICUBIC)
    padded = Image.new("RGB", (size, size), fill)
    padded.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
    return padded

crop = resize_pad(query_img)   # 224×224, white-padded

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(query_img); axes[0].set_title(f"Uploaded  {query_img.size[0]}×{query_img.size[1]}"); axes[0].axis("off")
axes[1].imshow(crop);      axes[1].set_title("Resized + padded  224×224");                        axes[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
### 3b — Extract query features

query_tensor = preprocess(crop).unsqueeze(0).to(DEVICE)   # (1, 3, 224, 224)

with torch.no_grad():
    query_feats = dino.forward_features(query_tensor)["x_norm_patchtokens"]  # (1, 256, 1024)

query_feats = F.normalize(query_feats.squeeze(0), dim=-1)  # (256, 1024)
print(f"Query features: {tuple(query_feats.shape)}")

In [ ]:
### 3c — Per-patch nearest-neighbour retrieval + template scoring

# Load cached template features (skip if already in memory from Section 2)
if "template_feats" not in dir():
    template_feats = torch.load("templates/template_features.pt")

tf = template_feats.to(DEVICE)    # (162, 256, 1024)
qf = query_feats.to(DEVICE)       # (256, 1024)

# Flatten all template patches: (162*256, 1024)
tf_flat = tf.view(162 * 256, 1024)

# Cosine similarity: each query patch vs every template patch
# Both are L2-normalised → dot product = cosine sim
sim_matrix = qf @ tf_flat.T       # (256, 41472)

# For each query patch: best matching template patch (globally)
best_sim, best_idx = sim_matrix.max(dim=1)     # (256,), (256,)
best_template = best_idx // 256                # which of 162 templates

# Apply 0.5 threshold (paper Eq. 2)
keep = best_sim >= 0.5
kept_sims = best_sim[keep]
kept_templates = best_template[keep]
print(f"{keep.sum().item()} / 256 query patches passed the 0.5 threshold")

# Score each template: mean similarity over its assigned query patches (paper Eq. 3)
scores = torch.zeros(162, device=DEVICE)
counts = torch.zeros(162, device=DEVICE)
scores.scatter_add_(0, kept_templates, kept_sims)
counts.scatter_add_(0, kept_templates, torch.ones_like(kept_sims))
template_scores = scores / counts.clamp(min=1)

# Top-5 results
K = 5
topk = template_scores.topk(K)
print(f"\nTop-{K} retrieved templates:")
for rank, (score, idx) in enumerate(zip(topk.values, topk.indices), 1):
    row = df.iloc[idx.item()]
    print(f"  #{rank}  template {idx.item():3d}  score={score:.3f}  "
          f"az={row['azimuth_deg']:+7.1f}°  el={row['elevation_deg']:+6.1f}°")

In [ ]:
### 3d — Visualisation

best_i = topk.indices[0].item()
best_row = df.iloc[best_i]

fig, axes = plt.subplots(1, K + 1, figsize=(3 * (K + 1), 3))

axes[0].imshow(crop)
axes[0].set_title("Query crop", fontsize=9)
axes[0].axis("off")

for rank, idx in enumerate(topk.indices, 1):
    row = df.iloc[idx.item()]
    axes[rank].imshow(templates_rgb[idx.item()])
    axes[rank].set_title(
        f"#{rank}  score={template_scores[idx]:.2f}\n"
        f"az={row['azimuth_deg']:+.0f}°  el={row['elevation_deg']:+.0f}°",
        fontsize=8,
    )
    axes[rank].axis("off")

plt.suptitle("Query  →  Top-5 retrieved templates", fontsize=11)
plt.tight_layout(); plt.show()

print(f"\nPredicted viewpoint:")
print(f"  Azimuth  : {best_row['azimuth_deg']:+.1f}°")
print(f"  Elevation: {best_row['elevation_deg']:+.1f}°")
print(f"  R_world_to_cam:\n{np.array(best_row['R_world_to_cam_3x3'])}")

---

## 4 — Train Fist (Stage 2)

**Fist** predicts the 4 remaining DoF after Stage 1 fixes the out-of-plane viewpoint:
- `(cos α, sin α)` — in-plane rotation
- `(tx, ty)` — 2D translation (normalised, [-0.15, 0.15])
- `log s` — scale (log-space for symmetric distribution)

### Architecture
- **Backbone:** ResNet18 truncated at `layer3` → outputs `(256, 16, 16)` dense features given **256×256** input (`256 / 2⁴ = 16`)
- **Per-correspondence MLP:** `concat(query_feat, template_feat) = 512 → 256 → 5`

### Training data
We have only the 162 templates of one object — no real query pairs. So we generate synthetic pairs by warping each template with a known random affine `(α, s, tx, ty)` to create a "fake query" Q. Fist learns to recover that known affine from `(T, Q)` features.

- **Augmentation ranges:** α ∈ [-180°, 180°], s ∈ [0.7, 1.3], (tx, ty) ∈ [-15%, 15%]
- **Loss:** L1 on the 5-vector `(cos α, sin α, tx, ty, log s)`
- **Optimizer:** Adam lr=1e-3, ~30 epochs × 100 steps × batch 32 (~25 min on T4)

In [ ]:
### 4a — Define Fist model

import torch.nn as nn
from torchvision.models import resnet18

class FistBackbone(nn.Module):
    """ResNet18 truncated at layer3 → (B, 256, 16, 16) given 256×256 input."""
    def __init__(self):
        super().__init__()
        m = resnet18(weights=None)
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool)
        self.layer1, self.layer2, self.layer3 = m.layer1, m.layer2, m.layer3

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        return x   # (B, 256, 16, 16)

class FistMLP(nn.Module):
    """Per-correspondence head: concat(q,t)=512 → 256 → 5."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 5),   # cos α, sin α, tx, ty, log s
        )

    def forward(self, qt_concat):
        return self.net(qt_concat)

class Fist(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = FistBackbone()
        self.mlp = FistMLP()

# Quick shape sanity check
_test = Fist().to(DEVICE)
with torch.no_grad():
    _x = torch.randn(2, 3, 256, 256, device=DEVICE)
    _f = _test.backbone(_x)
print(f"Backbone output shape: {tuple(_f.shape)}  (expect (2, 256, 16, 16))")
del _test, _x, _f

In [ ]:
### 4b — Synthetic pair generator

import kornia.geometry.transform as KT

FIST_SIZE = 256

fist_preprocess = T.Compose([
    T.Resize((FIST_SIZE, FIST_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Pre-load all 162 templates as a single tensor on GPU
templates_tensor = torch.stack([fist_preprocess(img) for img in templates_rgb]).to(DEVICE)
print(f"Templates tensor: {tuple(templates_tensor.shape)}  ({templates_tensor.element_size() * templates_tensor.nelement() / 1e6:.1f} MB)")

def sample_affine(B, device):
    """α ∈ [-π, π],  s ∈ [0.7, 1.3],  (tx, ty) ∈ [-0.15·SIZE, 0.15·SIZE] in pixels."""
    alpha = (torch.rand(B, device=device) * 2 - 1) * np.pi
    s     =  torch.rand(B, device=device) * 0.6 + 0.7
    tx    = (torch.rand(B, device=device) * 2 - 1) * 0.15 * FIST_SIZE
    ty    = (torch.rand(B, device=device) * 2 - 1) * 0.15 * FIST_SIZE
    return alpha, s, tx, ty

def affine_matrix(alpha, s, tx, ty):
    """2×3 affine that rotates around image centre, scales, then translates.
    Maps source pixel → destination pixel: dst = R*s*(src - c) + c + (tx, ty)."""
    cos_a, sin_a = torch.cos(alpha), torch.sin(alpha)
    c = FIST_SIZE / 2
    M = torch.zeros(alpha.shape[0], 2, 3, device=alpha.device)
    M[:, 0, 0] =  s * cos_a;  M[:, 0, 1] = -s * sin_a
    M[:, 1, 0] =  s * sin_a;  M[:, 1, 1] =  s * cos_a
    M[:, 0, 2] = c - s * (cos_a * c - sin_a * c) + tx
    M[:, 1, 2] = c - s * (sin_a * c + cos_a * c) + ty
    return M

def make_batch(B=32):
    """Sample B templates → generate B (T, Q) pairs with known affines."""
    idx = torch.randint(0, len(templates_tensor), (B,), device=DEVICE)
    T_batch = templates_tensor[idx]                               # (B, 3, 256, 256)
    alpha, s, tx, ty = sample_affine(B, DEVICE)
    M = affine_matrix(alpha, s, tx, ty)
    Q_batch = KT.warp_affine(T_batch, M, dsize=(FIST_SIZE, FIST_SIZE), padding_mode="border")
    targets = torch.stack([
        torch.cos(alpha), torch.sin(alpha),
        tx / FIST_SIZE, ty / FIST_SIZE,
        torch.log(s),
    ], dim=1)                                                     # (B, 5)
    return T_batch, Q_batch, targets

# Visual sanity check
T_b, Q_b, tgt = make_batch(B=4)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
def denorm(t):
    img = t.cpu().permute(1, 2, 0).numpy() * std + mean
    return img.clip(0, 1)
for i in range(4):
    axes[0, i].imshow(denorm(T_b[i])); axes[0, i].set_title(f"T  {i}");                axes[0, i].axis("off")
    a_deg = np.degrees(np.arctan2(tgt[i, 1].item(), tgt[i, 0].item()))
    s_v   = np.exp(tgt[i, 4].item())
    axes[1, i].imshow(denorm(Q_b[i]))
    axes[1, i].set_title(f"Q  α={a_deg:+.0f}° s={s_v:.2f}", fontsize=8)
    axes[1, i].axis("off")
plt.suptitle("Synthetic training pairs (T → Q with known affine)"); plt.tight_layout(); plt.show()

In [ ]:
### 4c — Train Fist (~25 min on T4)

import time

fist = Fist().to(DEVICE)
opt = torch.optim.Adam(fist.parameters(), lr=1e-3)
loss_fn = nn.L1Loss()

EPOCHS = 30
STEPS_PER_EPOCH = 100
BATCH = 32

NUM_PATCHES = 16 * 16   # 256

start = time.time()
losses = []
for epoch in range(EPOCHS):
    fist.train()
    epoch_loss = 0.0
    for step in range(STEPS_PER_EPOCH):
        T_batch, Q_batch, targets = make_batch(BATCH)
        F_T = fist.backbone(T_batch)                              # (B, 256, 16, 16)
        F_Q = fist.backbone(Q_batch)
        # Flatten spatial → (B, num_patches, dim)
        F_T_flat = F_T.permute(0, 2, 3, 1).reshape(BATCH, NUM_PATCHES, 256)
        F_Q_flat = F_Q.permute(0, 2, 3, 1).reshape(BATCH, NUM_PATCHES, 256)
        # Identity correspondence: same (i,j) in T and Q (single global affine)
        qt = torch.cat([F_Q_flat, F_T_flat], dim=-1)              # (B, 256, 512)
        preds = fist.mlp(qt)                                       # (B, 256, 5)
        target_expanded = targets.unsqueeze(1).expand(-1, NUM_PATCHES, -1)
        loss = loss_fn(preds, target_expanded)
        opt.zero_grad(); loss.backward(); opt.step()
        epoch_loss += loss.item()
    epoch_loss /= STEPS_PER_EPOCH
    losses.append(epoch_loss)
    elapsed = time.time() - start
    print(f"epoch {epoch+1:2d}/{EPOCHS}  loss={epoch_loss:.4f}  elapsed={elapsed:.0f}s")

torch.save(fist.state_dict(), "templates/fist_best.pt")
print(f"\nSaved templates/fist_best.pt  (persists to Drive via symlink)")

# Loss curve
plt.figure(figsize=(6, 3))
plt.plot(range(1, EPOCHS + 1), losses, marker="o", markersize=3)
plt.xlabel("Epoch"); plt.ylabel("L1 loss"); plt.title("Fist training loss")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---

## 5 — Stage 2 Inference + Full 6-DoF Pose

Combines Stage 1's matched template with Fist's per-correspondence affine predictions into a full 6-DoF pose.

```
Stage 1: best template + per-patch correspondences (256 query→template pairs)
   ↓
Fist on (query_crop_256, best_template_256)  →  (256, 16, 16) features each
   ↓
Per Stage-1 correspondence → MLP → predicted (cos α, sin α, tx, ty, log s)
   ↓
RANSAC over the affine predictions → robust global (α*, s*, tx*, ty*)
   ↓
Closed-form lift:
  R_final = R_inplane(α*) ∘ R_template_world_to_cam
  t_final = backproject((tx*, ty*), depth = template_distance / s*)
   ↓
Output: 3×3 rotation matrix + 3D translation vector
```

In [ ]:
### 5a — Camera intrinsics (manual)

# ── Replace with your camera's actual values for accurate 3D translation. ─────
# Defaults assume ~70° horizontal FOV (typical phone main camera).
img_w, img_h = query_img.size
fx = img_w * 0.7        # focal length (px)
fy = img_w * 0.7
cx = img_w / 2          # principal point
cy = img_h / 2

K = np.array([[fx,  0, cx],
              [ 0, fy, cy],
              [ 0,  0,  1]], dtype=np.float64)
print(f"Intrinsics: fx={fx:.1f}  fy={fy:.1f}  cx={cx:.1f}  cy={cy:.1f}")
print(f"K =\n{K}")

In [ ]:
### 5b — Load Fist + run on Stage-1 correspondences

# Load trained Fist (skip if already in memory)
if "fist" not in dir() or not isinstance(fist, Fist):
    fist = Fist().to(DEVICE)
    fist.load_state_dict(torch.load("templates/fist_best.pt", map_location=DEVICE))
fist.eval()

best_idx_template = topk.indices[0].item()
best_template_pil = templates_rgb[best_idx_template]
print(f"Using top-1 template: idx={best_idx_template}  "
      f"az={df.iloc[best_idx_template]['azimuth_deg']:+.1f}°  "
      f"el={df.iloc[best_idx_template]['elevation_deg']:+.1f}°")

# Encode both at 256×256 for Fist
T_in = fist_preprocess(best_template_pil).unsqueeze(0).to(DEVICE)   # (1, 3, 256, 256)
Q_in = fist_preprocess(crop).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    F_T = fist.backbone(T_in).squeeze(0)   # (256, 16, 16)
    F_Q = fist.backbone(Q_in).squeeze(0)

F_T_flat = F_T.permute(1, 2, 0).reshape(NUM_PATCHES, 256)   # (256_patches, 256_dim)
F_Q_flat = F_Q.permute(1, 2, 0).reshape(NUM_PATCHES, 256)

# Reuse Stage 1's correspondences:
#   `best_template` (256,) — which template each query patch matched to
#   `best_idx`      (256,) — which template-patch index in the flattened (162×256) bank
patches_in_best = (best_template == best_idx_template)
n_corr = patches_in_best.sum().item()
print(f"{n_corr}/256 query patches matched the top-1 template")

if n_corr < 5:
    raise RuntimeError("Too few Stage-1 correspondences for the top template — "
                       "Stage 1 retrieval is weak. Re-check the query crop.")

q_patch_idx = torch.nonzero(patches_in_best).squeeze(1)             # (n_corr,)
t_patch_idx = (best_idx[patches_in_best] % 256).to(torch.long)      # (n_corr,)

with torch.no_grad():
    q_feats = F_Q_flat[q_patch_idx]                                  # (n_corr, 256)
    t_feats = F_T_flat[t_patch_idx]
    preds = fist.mlp(torch.cat([q_feats, t_feats], dim=-1))         # (n_corr, 5)
preds_np = preds.cpu().numpy()
print(f"Fist preds: shape={preds_np.shape}")

In [ ]:
### 5c — RANSAC robust aggregation

def affine_from_pred(p):
    """Decode raw prediction → (cos α, sin α, s, tx, ty) with normalised cos/sin."""
    cos_a, sin_a, tx, ty, log_s = p
    norm = np.hypot(cos_a, sin_a) + 1e-8
    return cos_a / norm, sin_a / norm, np.exp(log_s), tx, ty

# Inlier thresholds
THR_ANGLE = np.deg2rad(15)   # rad
THR_SCALE = 0.15             # log-scale absolute diff
THR_TRANS = 0.06             # normalised translation distance

best_n_inliers = -1
best_inliers = None
n_trials = min(200, len(preds_np))

for trial in range(n_trials):
    h = preds_np[trial]
    cos_h, sin_h, s_h, tx_h, ty_h = affine_from_pred(h)
    angle_h = np.arctan2(sin_h, cos_h)
    log_s_h = np.log(s_h)

    inlier_mask = np.zeros(len(preds_np), dtype=bool)
    for i, p in enumerate(preds_np):
        c, s, sc, txp, typ = affine_from_pred(p)
        ang = np.arctan2(s, c)
        d_ang = np.abs(np.angle(np.exp(1j * (ang - angle_h))))
        d_s   = np.abs(np.log(sc) - log_s_h)
        d_t   = np.hypot(txp - tx_h, typ - ty_h)
        if d_ang < THR_ANGLE and d_s < THR_SCALE and d_t < THR_TRANS:
            inlier_mask[i] = True

    if inlier_mask.sum() > best_n_inliers:
        best_n_inliers = inlier_mask.sum()
        best_inliers   = inlier_mask

print(f"RANSAC: {best_n_inliers}/{len(preds_np)} inliers")

# Robust refit: median over inliers
inlier_data = np.array([affine_from_pred(p) for p in preds_np[best_inliers]])
cos_a = np.median(inlier_data[:, 0]); sin_a = np.median(inlier_data[:, 1])
norm  = np.hypot(cos_a, sin_a)
cos_a, sin_a = cos_a / norm, sin_a / norm
alpha_final = np.arctan2(sin_a, cos_a)
s_final  = np.median(inlier_data[:, 2])
tx_final = np.median(inlier_data[:, 3])
ty_final = np.median(inlier_data[:, 4])

print(f"\nRefined affine:")
print(f"  α  = {np.degrees(alpha_final):+7.1f}°")
print(f"  s  = {s_final:7.3f}")
print(f"  tx = {tx_final:+7.3f}  (normalised)")
print(f"  ty = {ty_final:+7.3f}  (normalised)")

In [ ]:
### 5d — Closed-form 6-DoF lift

# ── Final rotation: in-plane (camera-Z) composed with template's R ────────────
R_template = np.array(df.iloc[best_idx_template]["R_world_to_cam_3x3"], dtype=np.float64)
R_inplane = np.array([
    [np.cos(alpha_final), -np.sin(alpha_final), 0.0],
    [np.sin(alpha_final),  np.cos(alpha_final), 0.0],
    [0.0,                  0.0,                 1.0],
], dtype=np.float64)
R_final = R_inplane @ R_template

# Sanity check: orthogonal + det 1
ortho_err = np.linalg.norm(R_final @ R_final.T - np.eye(3))
det_err   = abs(np.linalg.det(R_final) - 1.0)
print(f"Rotation sanity: ortho_err={ortho_err:.2e}  det_err={det_err:.2e}")

# ── Final 3D translation ──────────────────────────────────────────────────────
# Template was rendered with camera at known distance from object centre (origin)
template_distance = float(np.linalg.norm(df.iloc[best_idx_template]["t_cam_to_world_3"]))

# Depth: closer object → larger projected size → smaller s in template space
z = template_distance / s_final

# tx_final, ty_final are normalised offsets ([-0.15, 0.15]) in 256×256 Fist crop.
# Convert to query-image pixel offsets by scaling against image dimensions, then
# unproject (u_obj, v_obj) at depth z through K.
u_obj = cx + tx_final * img_w
v_obj = cy + ty_final * img_h
x_world = (u_obj - cx) * z / fx
y_world = (v_obj - cy) * z / fy
t_final = np.array([x_world, y_world, z], dtype=np.float64)

# ── Print final pose ──────────────────────────────────────────────────────────
print(f"\n══ Predicted 6-DoF pose ══")
print(f"R (3×3):")
for row in R_final:
    print(f"  [{row[0]:+8.4f}  {row[1]:+8.4f}  {row[2]:+8.4f}]")
print(f"t (3,):  [{t_final[0]:+8.4f}  {t_final[1]:+8.4f}  {t_final[2]:+8.4f}]   (object-frame units)")
print(f"\nDecomposed:")
print(f"  Out-of-plane (from template):  az={df.iloc[best_idx_template]['azimuth_deg']:+.1f}°   "
      f"el={df.iloc[best_idx_template]['elevation_deg']:+.1f}°")
print(f"  In-plane (from Fist):          α ={np.degrees(alpha_final):+.1f}°")
print(f"  Depth (from scale):            z ={z:.3f}  (template_distance={template_distance:.3f}, s={s_final:.3f})")

In [ ]:
### 5e — Visualisation: warp template by predicted affine

# If Fist worked, applying the predicted affine to the template should
# produce something resembling the query crop.
M_pred = affine_matrix(
    torch.tensor([alpha_final], dtype=torch.float32, device=DEVICE),
    torch.tensor([s_final],     dtype=torch.float32, device=DEVICE),
    torch.tensor([tx_final * FIST_SIZE], dtype=torch.float32, device=DEVICE),
    torch.tensor([ty_final * FIST_SIZE], dtype=torch.float32, device=DEVICE),
)
T_warped = KT.warp_affine(T_in, M_pred, dsize=(FIST_SIZE, FIST_SIZE), padding_mode="border")
T_warped_img = denorm(T_warped.squeeze(0))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(crop);              axes[0].set_title("Query crop");                    axes[0].axis("off")
axes[1].imshow(best_template_pil); axes[1].set_title(f"Best template (idx {best_idx_template})"); axes[1].axis("off")
axes[2].imshow(T_warped_img);      axes[2].set_title(f"Template ∘ predicted affine\nα={np.degrees(alpha_final):+.1f}° s={s_final:.2f}"); axes[2].axis("off")
plt.suptitle("Stage 2 — sanity check: warped template should resemble query"); plt.tight_layout(); plt.show()